In [1]:
# =============================================================================
# CELL 1: Cài đặt thư viện
# =============================================================================
import subprocess, sys
print("Đang cài đặt các thư viện cần thiết...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow', 'polars', 'scikit-learn', 'duckdb'])


Đang cài đặt các thư viện cần thiết...


0

In [2]:
# =============================================================================
# CELL 2: Import & Cấu hình đường dẫn
# =============================================================================
import os, gc, pickle
import numpy as np
import pandas as pd
import polars as pl
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as Data
from tqdm.auto import tqdm
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Thiết bị huấn luyện: {device}")
IS_KAGGLE = os.path.exists('/kaggle')
INPUT_DIR = '/kaggle/input/datasets/b22dckh068donngkhoa/deep-learning-dataset' if IS_KAGGLE else os.path.abspath('./')
WORKING_DIR = '/kaggle/working/' if IS_KAGGLE else os.path.abspath('./')
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"WORKING_DIR: {WORKING_DIR}")
FEAT_PATH = os.path.join(INPUT_DIR, 'features.parquet')
DICT_OUT = os.path.join(WORKING_DIR, 'fmlp_category_dict.pkl')
SCALER_OUT = os.path.join(WORKING_DIR, 'fmlp_scaler.pkl')
MODEL_OUT = os.path.join(WORKING_DIR, 'fmlp_model.pth')
CONFIG_OUT = os.path.join(WORKING_DIR, 'fmlp_config.pkl')
META_PATH  = os.path.join(INPUT_DIR, 'filtered_metadata.parquet')
TRAIN_PATH = os.path.join(INPUT_DIR, 'train_interactions.parquet')
CAND_PATH  = os.path.join(INPUT_DIR, 'candidates_phase2.parquet')
TEST_PATH  = os.path.join(INPUT_DIR, 'test_interactions.parquet')
TMP_SCORE_PATH = os.path.join(WORKING_DIR, 'tmp_scores.parquet')
FINAL_TOP100_PATH = os.path.join(WORKING_DIR, 'top100_final_recommendations.parquet')

Thiết bị huấn luyện: cuda
INPUT_DIR: /kaggle/input/datasets/b22dckh068donngkhoa/deep-learning-dataset
WORKING_DIR: /kaggle/working/


In [3]:
# =============================================================================
# CELL 3: Nạp dữ liệu & tính vocabulary
# =============================================================================
print("Đang nạp file đặc trưng features.parquet bằng Polars...")
df_pl = pl.read_parquet(FEAT_PATH)
print(f"Kích thước dữ liệu: {df_pl.height:,} dòng, {df_pl.width} cột")
print("Đang tính toán kích thước vocabulary toàn cục cho User ID và Item ID...")
try:
    train_df = pl.scan_parquet(TRAIN_PATH)
    test_df = pl.scan_parquet(TEST_PATH)
    max_user_id = max(
        train_df.select(pl.col('mapped_user_id').max()).collect().item(),
        test_df.select(pl.col('mapped_user_id').max()).collect().item()
    )
    max_item_id = max(
        train_df.select(pl.col('mapped_item_id').max()).collect().item(),
        test_df.select(pl.col('mapped_item_id').max()).collect().item()
    )
    num_users = int(max_user_id) + 1
    num_items = int(max_item_id) + 1
    print(f"Kích thước danh mục: Users={num_users:,}, Items={num_items:,}")
except Exception as e:
    print("Không tìm thấy train/test_interactions.parquet, dùng kích thước trong features.parquet:", e)
    num_users = int(df_pl['mapped_user_id'].max()) + 1
    num_items = int(df_pl['mapped_item_id'].max()) + 1
    print(f"Kích thước danh mục (local): Users={num_users:,}, Items={num_items:,}")


Đang nạp file đặc trưng features.parquet bằng Polars...
Kích thước dữ liệu: 24,446,096 dòng, 17 cột
Đang tính toán kích thước vocabulary toàn cục cho User ID và Item ID...
Kích thước danh mục: Users=2,257,154, Items=626,748


In [4]:
# =============================================================================
# CELL 4: Tiền xử lý đặc trưng
# =============================================================================
print("Đang thực hiện tiền xử lý đặc trưng...")
dense_features = [
    'user_total_actions', 
    'item_total_sales', 
    'price', 
    'average_rating',
    'rating_number',
    'user_avg_rating_given',
    'item_actual_avg_rating',  
    'item_verified_ratio',
    'item_log_helpful_votes',
    'store_popularity',
    'sasrec_score', 
    'lightgcn_score'
]
# 1. Điền NaN
df_pl = df_pl.with_columns([
    pl.col(feat).fill_null(0.0) for feat in dense_features
])
# 2. Mã hóa Label Encoding cho main_category
categories = df_pl.select('main_category').fill_null('Unknown').to_series().to_list()
le = LabelEncoder()
encoded_cats = le.fit_transform(categories)
df_pl = df_pl.with_columns(pl.Series('main_category', encoded_cats, dtype=pl.Int32))
le_dict = {'main_category': le}
with open(DICT_OUT, 'wb') as f:
    pickle.dump(le_dict, f)
print("-> Đã mã hóa và lưu category dict tại:", DICT_OUT)
# Chuyển sang Pandas để fit scaler
df = df_pl.to_pandas()
del df_pl
gc.collect()
# 3. Chia ngẫu nhiên phân tầng 80/20
y = df['label'].values
train_idx, val_idx = train_test_split(
    np.arange(len(df)), 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)
print(f"Số lượng mẫu Train: {len(train_idx):,}, Validation: {len(val_idx):,}")
# 4. Fit MinMaxScaler trên Train và transform trên Val
scaler = MinMaxScaler(feature_range=(0, 1))
df.loc[train_idx, dense_features] = scaler.fit_transform(df.iloc[train_idx][dense_features])
df.loc[val_idx, dense_features] = scaler.transform(df.iloc[val_idx][dense_features])
with open(SCALER_OUT, 'wb') as f:
    pickle.dump(scaler, f)
print("-> Đã chuẩn hóa và lưu scaler tại:", SCALER_OUT)
# Lưu cấu hình mô hình
config = {
    'num_users': num_users,
    'num_items': num_items,
    'main_category_classes': le.classes_,
    'dense_features': dense_features
}
with open(CONFIG_OUT, 'wb') as f:
    pickle.dump(config, f)
print("-> Đã lưu cấu hình mô hình tại:", CONFIG_OUT)

Đang thực hiện tiền xử lý đặc trưng...
-> Đã mã hóa và lưu category dict tại: /kaggle/working/fmlp_category_dict.pkl
Số lượng mẫu Train: 19,556,876, Validation: 4,889,220


/tmp/ipykernel_58/1262094559.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.00614251 0.00982801 0.         ... 0.002457   0.         0.00859951]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df.loc[train_idx, dense_features] = scaler.fit_transform(df.iloc[train_idx][dense_features])
/tmp/ipykernel_58/1262094559.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.00369348 0.00407609 0.01149348 ... 0.00978261 0.00217174 0.00465652]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df.loc[train_idx, dense_features] = scaler.fit_transform(df.iloc[train_idx][dense_features])
/tmp/ipykernel_58/1262094559.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas

-> Đã chuẩn hóa và lưu scaler tại: /kaggle/working/fmlp_scaler.pkl
-> Đã lưu cấu hình mô hình tại: /kaggle/working/fmlp_config.pkl


In [5]:
# =============================================================================
# CELL 5: BPR Dataset & DataLoader
# =============================================================================
print("Chuẩn bị BPR Pairwise DataLoader...")

user_dense_cols = [
    'user_total_actions',
    'user_avg_rating_given',
    'sasrec_score',
    'lightgcn_score'
]

item_dense_cols = [
    'item_total_sales',
    'price',
    'average_rating',
    'rating_number',
    'item_actual_avg_rating',
    'item_verified_ratio',
    'item_log_helpful_votes',
    'store_popularity'
]

# ============================================================
# 1. Tạo tensor cho train
# ============================================================
train_item_ids = torch.tensor(
    df.iloc[train_idx]['mapped_item_id'].values,
    dtype=torch.long
)

train_cat_ids = torch.tensor(
    df.iloc[train_idx]['main_category'].values,
    dtype=torch.long
)

train_user_dense = torch.tensor(
    df.iloc[train_idx][user_dense_cols].values,
    dtype=torch.float32
)

train_item_dense = torch.tensor(
    df.iloc[train_idx][item_dense_cols].values,
    dtype=torch.float32
)

train_labels = y[train_idx]

train_user_ids_arr = df.iloc[train_idx]['mapped_user_id'].values


# ============================================================
# 2. Tạo tensor cho validation
# ============================================================
val_item_ids = torch.tensor(
    df.iloc[val_idx]['mapped_item_id'].values,
    dtype=torch.long
)

val_cat_ids = torch.tensor(
    df.iloc[val_idx]['main_category'].values,
    dtype=torch.long
)

val_user_dense = torch.tensor(
    df.iloc[val_idx][user_dense_cols].values,
    dtype=torch.float32
)

val_item_dense = torch.tensor(
    df.iloc[val_idx][item_dense_cols].values,
    dtype=torch.float32
)

val_labels = torch.tensor(
    y[val_idx],
    dtype=torch.float32
)


# ============================================================
# 3. Build mapping user -> positive / negative train indices
# ============================================================
print("Đang xây dựng index BPR (positive/negative per user)...")

pos_positions = np.where(train_labels == 1)[0]
neg_positions = np.where(train_labels == 0)[0]

pos_uids = train_user_ids_arr[pos_positions]
neg_uids = train_user_ids_arr[neg_positions]

user_pos_map = defaultdict(list)

for i, uid in enumerate(pos_uids):
    user_pos_map[uid].append(pos_positions[i])

user_neg_map = {}

neg_df_tmp = pd.DataFrame({
    'uid': neg_uids,
    'idx': neg_positions
})

for uid, group in neg_df_tmp.groupby('uid')['idx']:
    user_neg_map[uid] = group.values

del neg_df_tmp, pos_uids, neg_uids, pos_positions, neg_positions
gc.collect()

n_users_with_both = sum(1 for uid in user_pos_map if uid in user_neg_map)

print(f"Users có cả pos & neg: {n_users_with_both:,}")
print(f"Positive samples trong train: {len(train_labels[train_labels == 1]):,}")


# ============================================================
# 4. BPR Pairwise Dataset cho train
# ============================================================
class BPRPairDataset(Data.Dataset):
    """
    Mỗi sample gồm:
    - 1 positive item
    - 1 random negative item cùng user

    neg_per_pos: số negative được sample cho mỗi positive trong một epoch.
    """
    def __init__(
        self,
        item_ids,
        cat_ids,
        user_dense,
        item_dense,
        user_pos_map,
        user_neg_map,
        neg_per_pos=5
    ):
        self.item_ids = item_ids
        self.cat_ids = cat_ids
        self.user_dense = user_dense
        self.item_dense = item_dense
        self.user_neg_map = user_neg_map

        self.samples = []

        for uid, pos_indices in user_pos_map.items():
            if uid in user_neg_map:
                for pos_idx in pos_indices:
                    for _ in range(neg_per_pos):
                        self.samples.append((uid, pos_idx))

        print(
            f"BPR Dataset: {len(self.samples):,} pairs/epoch "
            f"(neg_per_pos={neg_per_pos})"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        uid, pos_idx = self.samples[idx]

        neg_candidates = self.user_neg_map[uid]
        neg_idx = neg_candidates[np.random.randint(len(neg_candidates))]

        return (
            # Positive pair
            self.item_ids[pos_idx],
            self.cat_ids[pos_idx],
            self.user_dense[pos_idx],
            self.item_dense[pos_idx],

            # Negative pair
            self.item_ids[neg_idx],
            self.cat_ids[neg_idx],
            self.user_dense[neg_idx],
            self.item_dense[neg_idx],
        )


# ============================================================
# 5. Pointwise Dataset cho validation AUC
# ============================================================
class PointwiseDataset(Data.Dataset):
    def __init__(self, item_ids, cat_ids, user_dense, item_dense, labels):
        self.item_ids = item_ids
        self.cat_ids = cat_ids
        self.user_dense = user_dense
        self.item_dense = item_dense
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.item_ids[idx],
            self.cat_ids[idx],
            self.user_dense[idx],
            self.item_dense[idx],
            self.labels[idx]
        )


# ============================================================
# 6. Tạo Dataset
# ============================================================
NEG_PER_POS = 20

train_bpr_dataset = BPRPairDataset(
    train_item_ids,
    train_cat_ids,
    train_user_dense,
    train_item_dense,
    user_pos_map,
    user_neg_map,
    neg_per_pos=NEG_PER_POS
)

val_dataset = PointwiseDataset(
    val_item_ids,
    val_cat_ids,
    val_user_dense,
    val_item_dense,
    val_labels
)


# ============================================================
# 7. Tạo DataLoader
# ============================================================
batch_size = 16384

train_loader = Data.DataLoader(
    train_bpr_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device == 'cuda')
)

val_loader = Data.DataLoader(
    val_dataset,
    batch_size=batch_size * 2,
    shuffle=False,
    num_workers=0,
    pin_memory=(device == 'cuda')
)


# ============================================================
# 8. Giải phóng bộ nhớ
# ============================================================
print("Giải phóng bộ nhớ...")

del df
del train_user_ids_arr
del train_labels

gc.collect()

print("Cell 5 hoàn tất!")

Chuẩn bị BPR Pairwise DataLoader...
Đang xây dựng index BPR (positive/negative per user)...
Users có cả pos & neg: 66,849
Positive samples trong train: 74,894
BPR Dataset: 1,497,880 pairs/epoch (neg_per_pos=20)
Giải phóng bộ nhớ...
Cell 5 hoàn tất!


In [6]:
# =============================================================================
# CELL 6: Định nghĩa mô hình FinalMLP (bỏ user_emb, trả logits)
# =============================================================================
class FeatureGate(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.gate(x)


class FinalMLP(nn.Module):
    def __init__(
        self,
        num_items,
        num_categories,
        item_emb_dim=8,
        cat_emb_dim=8,
        hidden_units=32,
        dropout=0.3
    ):
        super().__init__()

        # Giữ item embedding nhưng giảm rất nhỏ để tránh overfit
        self.item_emb = nn.Embedding(num_items, item_emb_dim)
        self.cat_emb = nn.Embedding(num_categories, cat_emb_dim)

        nn.init.normal_(self.item_emb.weight, mean=0.0, std=0.01)
        nn.init.normal_(self.cat_emb.weight, mean=0.0, std=0.01)

        # user_dense = ['user_total_actions', 'user_avg_rating_given', 'sasrec_score', 'lightgcn_score']
        user_input_dim = 4

        # item_dense có 8 feature
        item_input_dim = item_emb_dim + cat_emb_dim + 8

        self.user_gate = FeatureGate(user_input_dim)
        self.item_gate = FeatureGate(item_input_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(user_input_dim, hidden_units),
            nn.BatchNorm1d(hidden_units),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_units, hidden_units),
            nn.BatchNorm1d(hidden_units),
            nn.ReLU()
        )

        self.item_mlp = nn.Sequential(
            nn.Linear(item_input_dim, hidden_units),
            nn.BatchNorm1d(hidden_units),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_units, hidden_units),
            nn.BatchNorm1d(hidden_units),
            nn.ReLU()
        )

        self.W = nn.Parameter(torch.randn(hidden_units, hidden_units))
        nn.init.xavier_normal_(self.W)

        self.fc = nn.Linear(hidden_units * 2, 1)

        # Học trọng số phối hợp SASRec và LightGCN
        self.base_weight = nn.Parameter(torch.tensor([1.0, 1.0]))

        # Khống chế residual để MLP không phá baseline
        self.residual_alpha = nn.Parameter(torch.tensor(0.0))

    def forward(self, item_ids, cat_ids, user_dense, item_dense):
        i_e = self.item_emb(item_ids)
        c_e = self.cat_emb(cat_ids)

        user_in = user_dense
        item_in = torch.cat([i_e, c_e, item_dense], dim=-1)

        user_gated = self.user_gate(user_in)
        item_gated = self.item_gate(item_in)

        h_u = self.user_mlp(user_gated)
        h_i = self.item_mlp(item_gated)

        bilinear = torch.sum(torch.matmul(h_u, self.W) * h_i, dim=-1)
        concat_out = torch.cat([h_u, h_i], dim=-1)

        mlp_score = self.fc(concat_out).squeeze(-1) + bilinear

        # user_dense[:, 2] = sasrec_score
        # user_dense[:, 3] = lightgcn_score
        w = torch.softmax(self.base_weight, dim=0)
        base_score = w[0] * user_dense[:, 2] + w[1] * user_dense[:, 3]

        # Residual bị giới hạn biên độ, tránh làm hỏng rank gốc
        residual_scale = 0.2 * torch.sigmoid(self.residual_alpha)
        logits = base_score + residual_scale * torch.tanh(mlp_score)

        return logits
        
num_categories = len(le.classes_)
model = FinalMLP(num_items, num_categories).to(device)
print(model)


FinalMLP(
  (item_emb): Embedding(626748, 8)
  (cat_emb): Embedding(276, 8)
  (user_gate): FeatureGate(
    (gate): Sequential(
      (0): Linear(in_features=4, out_features=4, bias=True)
      (1): Sigmoid()
    )
  )
  (item_gate): FeatureGate(
    (gate): Sequential(
      (0): Linear(in_features=24, out_features=24, bias=True)
      (1): Sigmoid()
    )
  )
  (user_mlp): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=32, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
  )
  (item_mlp): Sequential(
    (0): Linear(in_features=24, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4):

In [7]:
# =============================================================================
# CELL 7: Huấn luyện BPR Pairwise
# =============================================================================
def bpr_loss(pos_scores, neg_scores):
    """BPR Loss: -log(sigmoid(score_pos - score_neg))"""
    return -F.logsigmoid(pos_scores - neg_scores).mean()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)
epochs = 10
best_auc = 0.0
patience_counter = 0
max_patience = 3
print("Bắt đầu huấn luyện FinalMLP với BPR Pairwise Loss...")
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for (pos_i_ids, pos_c_ids, pos_u_dense, pos_i_dense,
         neg_i_ids, neg_c_ids, neg_u_dense, neg_i_dense) in pbar:
        
        pos_i_ids  = pos_i_ids.to(device)
        pos_c_ids  = pos_c_ids.to(device)
        pos_u_dense = pos_u_dense.to(device)
        pos_i_dense = pos_i_dense.to(device)
        neg_i_ids  = neg_i_ids.to(device)
        neg_c_ids  = neg_c_ids.to(device)
        neg_u_dense = neg_u_dense.to(device)
        neg_i_dense = neg_i_dense.to(device)
        
        optimizer.zero_grad()
        
        pos_scores = model(pos_i_ids, pos_c_ids, pos_u_dense, pos_i_dense)
        neg_scores = model(neg_i_ids, neg_c_ids, neg_u_dense, neg_i_dense)
        
        loss = bpr_loss(pos_scores, neg_scores)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({'bpr_loss': f"{loss.item():.4f}"})
    
    epoch_loss /= n_batches
    
    # Validation — pointwise AUC (vẫn dùng để monitor)
    model.eval()
    val_preds = []
    val_targets = []
    with torch.no_grad():
        for i_ids, c_ids, u_dense, i_dense, labels in val_loader:
            i_ids  = i_ids.to(device)
            c_ids  = c_ids.to(device)
            u_dense = u_dense.to(device)
            i_dense = i_dense.to(device)
            
            logits = model(i_ids, c_ids, u_dense, i_dense)
            val_preds.extend(torch.sigmoid(logits).cpu().numpy())
            val_targets.extend(labels.numpy())
    
    val_auc = roc_auc_score(val_targets, val_preds)
    scheduler.step(val_auc)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1} - BPR Loss: {epoch_loss:.4f} - Val AUC: {val_auc:.4f} - LR: {current_lr:.6f}")
    
    if val_auc > best_auc:
        best_auc = val_auc
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_OUT)
        print(f"-> Đã lưu model tốt nhất với Val AUC: {best_auc:.4f}")
    else:
        patience_counter += 1
        print(f"-> Không cải thiện ({patience_counter}/{max_patience})")
        if patience_counter >= max_patience:
            print("Early stopping!")
            break
del y, train_idx, val_idx
gc.collect()

Bắt đầu huấn luyện FinalMLP với BPR Pairwise Loss...


Epoch 1/10:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 1 - BPR Loss: 0.5995 - Val AUC: 0.7060 - LR: 0.000200
-> Đã lưu model tốt nhất với Val AUC: 0.7060


Epoch 2/10:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 2 - BPR Loss: 0.5740 - Val AUC: 0.7068 - LR: 0.000200
-> Đã lưu model tốt nhất với Val AUC: 0.7068


...

Epoch 6/10:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 6 - BPR Loss: 0.5291 - Val AUC: 0.7066 - LR: 0.000200
-> Không cải thiện (2/3)


Epoch 7/10:   0%|          | 0/92 [00:00<?, ?it/s]

Epoch 7 - BPR Loss: 0.5265 - Val AUC: 0.7063 - LR: 0.000100
-> Không cải thiện (3/3)
Early stopping!


0

In [8]:
# =============================================================================
# CELL 8: Nạp lại model tốt nhất & chuẩn bị scoring
# =============================================================================
print("Đang nạp lại trọng số mô hình tốt nhất...")
model.load_state_dict(torch.load(MODEL_OUT, map_location=device))
model.eval()
print("Đang chuẩn bị đặc trưng Item và User cho bước Scoring...")
lf_train = pl.scan_parquet(TRAIN_PATH)
lf_meta = pl.scan_parquet(META_PATH)
df_use = (
    lf_train.group_by('mapped_user_id').agg([
        pl.len().alias('user_total_actions').cast(pl.Float32),
        pl.col('rating').mean().alias('user_avg_rating_given').cast(pl.Float32)
    ]).collect()
)
df_ite = (
    lf_train.group_by('mapped_item_id').agg([
        pl.len().alias('item_total_sales').cast(pl.Float32),
        pl.col('rating').mean().alias('item_actual_avg_rating').cast(pl.Float32),
        pl.col('verified_purchase').cast(pl.Float32).sum().alias('item_verified_sales'),
        pl.col('helpful_vote').cast(pl.Float32).sum().alias('item_raw_helpful_votes')
    ]).with_columns([
        (pl.col('item_verified_sales') / pl.col('item_total_sales')).fill_null(0.0).alias('item_verified_ratio'),
        (pl.col('item_raw_helpful_votes') + 1.0).log().alias('item_log_helpful_votes')
    ]).drop(['item_verified_sales', 'item_raw_helpful_votes']).collect()
)
store_counts = lf_meta.group_by('store').agg(pl.len().cast(pl.Float32).alias('store_popularity')).collect()
df_meta_feats = (
    lf_meta.select(['mapped_item_id', 'price', 'average_rating', 'rating_number', 'store', 'categories'])
    .with_columns([
        pl.col('price').cast(pl.Utf8).str.replace_all(r'[^0-9.]', '').cast(pl.Float32, strict=False),
        pl.col('categories').cast(pl.Utf8)
          .str.replace_all(r"\\[|\\]|'|\\\"", "")
          .str.split(',')
          .list.get(2)
          .str.strip_chars()
          .fill_null('Unknown')
          .cast(pl.String)
          .alias('main_category')
    ])
    .collect()
    .join(store_counts, on='store', how='left') 
    .drop(['store', 'categories'])              
    .unique(subset=['mapped_item_id'])
)
df_item_combined = df_meta_feats.join(df_ite, on='mapped_item_id', how='left')
categories_list = list(le.classes_)
cat_map_pl = pl.DataFrame({
    'main_category': categories_list,
    'main_category_encoded': list(range(len(categories_list)))
})
scale_values = scaler.scale_
min_values = scaler.min_
del lf_train, lf_meta, store_counts, df_ite, df_meta_feats
gc.collect()


Đang nạp lại trọng số mô hình tốt nhất...
Đang chuẩn bị đặc trưng Item và User cho bước Scoring...


0

In [9]:
# =============================================================================
# CELL 9: Scoring ứng viên bằng FinalMLP + Borda Fusion
# =============================================================================
import pyarrow.parquet as pq

print("Bắt đầu chấm điểm ứng viên bằng FinalMLP + Borda Fusion theo khối...")

# Trọng số cho Borda.
# 0.85 nghĩa là ưu tiên Borda 85%, FinalMLP chỉ chỉnh nhẹ 15%.
BORDA_WEIGHT = 0.85
FMLP_WEIGHT = 1.0 - BORDA_WEIGHT

pf = pq.ParquetFile(CAND_PATH)
reader = pf.iter_batches(batch_size=2000000)

writer = None

for batch in tqdm(reader, desc="Scoring Chunks"):
    chunk = pl.from_arrow(batch)

    # Join user/item features
    chunk = chunk.join(df_use, on='mapped_user_id', how='left')
    chunk = chunk.join(df_item_combined, on='mapped_item_id', how='left')

    # Tạo sasrec_score/lightgcn_score từ rank.
    # Rank càng nhỏ càng tốt => score càng lớn.
    chunk = chunk.with_columns([
        ((201.0 - pl.col('sasrec_rank').cast(pl.Float32)).clip(lower_bound=0.0))
            .fill_null(0.0)
            .alias('sasrec_score'),

        ((201.0 - pl.col('lightgcn_rank').cast(pl.Float32)).clip(lower_bound=0.0))
            .fill_null(0.0)
            .alias('lightgcn_score'),

        pl.col('average_rating').fill_null(3.0),
        pl.col('rating_number').fill_null(0.0),
        pl.col('user_total_actions').fill_null(0.0),
        pl.col('item_total_sales').fill_null(0.0),
        pl.col('user_avg_rating_given').fill_null(3.0),
        pl.col('item_actual_avg_rating').fill_null(3.0),
        pl.col('item_verified_ratio').fill_null(0.0),
        pl.col('item_log_helpful_votes').fill_null(0.0),
        pl.col('store_popularity').fill_null(0.0),
        pl.col('price').fill_null(0.0),
        pl.col('main_category').fill_null('Unknown')
    ])

    # Tính Borda score thô:
    # Nếu item xuất hiện cao trong SASRec/LightGCN thì điểm lớn.
    # Vì mỗi rank tối đa khoảng 200, tổng điểm tối đa khoảng 400.
    chunk = chunk.with_columns([
        (
            (pl.col('sasrec_score') + pl.col('lightgcn_score')) / 400.0
        )
        .clip(lower_bound=0.0, upper_bound=1.0)
        .alias('borda_score')
    ])

    # Encode category
    chunk = chunk.join(cat_map_pl, on='main_category', how='left')
    chunk = chunk.with_columns(
        pl.col('main_category_encoded').fill_null(0).alias('main_category')
    )

    # Chuẩn hóa dense features giống lúc train
    for idx, feat in enumerate(dense_features):
        chunk = chunk.with_columns(
            ((pl.col(feat).fill_null(0.0) * scale_values[idx]) + min_values[idx]).alias(feat)
        )

    # Lấy dữ liệu đưa vào FinalMLP
    chunk_pd = chunk.select(
        ['mapped_user_id', 'mapped_item_id', 'main_category', 'borda_score'] + dense_features
    ).to_pandas()

    i_ids = torch.tensor(chunk_pd['mapped_item_id'].values, dtype=torch.long, device=device)
    c_ids = torch.tensor(chunk_pd['main_category'].values, dtype=torch.long, device=device)
    u_dense = torch.tensor(chunk_pd[user_dense_cols].values, dtype=torch.float32, device=device)
    i_dense = torch.tensor(chunk_pd[item_dense_cols].values, dtype=torch.float32, device=device)

    # Scoring bằng FinalMLP
    model.eval()
    with torch.no_grad():
        scores_list = []
        inner_batch = 131072

        for start in range(0, len(i_ids), inner_batch):
            end = start + inner_batch

            logits = model(
                i_ids[start:end],
                c_ids[start:end],
                u_dense[start:end],
                i_dense[start:end]
            )

            # Sigmoid để đưa về [0, 1]
            batch_scores = torch.sigmoid(logits)
            scores_list.append(batch_scores.cpu().numpy())

        fmlp_scores = np.concatenate(scores_list)

    # Borda score cũng đang ở [0, 1]
    borda_scores = chunk_pd['borda_score'].values.astype(np.float32)

    # Fusion:
    # score cuối = 0.85 * Borda + 0.15 * FinalMLP
    final_scores = (
        BORDA_WEIGHT * borda_scores +
        FMLP_WEIGHT * fmlp_scores
    ).astype(np.float32)

    chunk_res = pl.DataFrame({
        'mapped_user_id': chunk['mapped_user_id'],
        'mapped_item_id': chunk['mapped_item_id'],
        'score': pl.Series(final_scores, dtype=pl.Float32),
        'borda_score': pl.Series(borda_scores, dtype=pl.Float32),
        'fmlp_score': pl.Series(fmlp_scores.astype(np.float32), dtype=pl.Float32)
    })

    table = chunk_res.to_arrow()

    if writer is None:
        writer = pq.ParquetWriter(TMP_SCORE_PATH, table.schema)

    writer.write_table(table)

    del chunk, chunk_pd, i_ids, c_ids, u_dense, i_dense
    del scores_list, fmlp_scores, borda_scores, final_scores, chunk_res, table
    gc.collect()

if writer is not None:
    writer.close()

print("Hoàn tất lưu điểm tạm thời FinalMLP + Borda Fusion!")
print(f"BORDA_WEIGHT = {BORDA_WEIGHT}, FMLP_WEIGHT = {FMLP_WEIGHT}")

Bắt đầu chấm điểm ứng viên bằng FinalMLP + Borda Fusion theo khối...


Scoring Chunks: 0it [00:00, ?it/s]

Hoàn tất lưu điểm tạm thời FinalMLP + Borda Fusion!
BORDA_WEIGHT = 0.85, FMLP_WEIGHT = 0.15000000000000002


In [10]:
# =============================================================================
# CELL 10: Trích xuất Top 100 bằng DuckDB
# =============================================================================
import duckdb
print("Đang thực hiện sắp xếp toàn cục để lấy Top 100 bằng DuckDB...")
duckdb.sql(f"""
    COPY (
        SELECT mapped_user_id, mapped_item_id, score
        FROM read_parquet('{TMP_SCORE_PATH}')
        QUALIFY ROW_NUMBER() OVER (PARTITION BY mapped_user_id ORDER BY score DESC, mapped_item_id ASC) <= 100
    ) TO '{FINAL_TOP100_PATH}' (FORMAT PARQUET);
""")
if os.path.exists(TMP_SCORE_PATH):
    os.remove(TMP_SCORE_PATH)
print(f"Hoàn tất! File gợi ý Top 100 cuối cùng đã được lưu tại: {FINAL_TOP100_PATH}")


Đang thực hiện sắp xếp toàn cục để lấy Top 100 bằng DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Hoàn tất! File gợi ý Top 100 cuối cùng đã được lưu tại: /kaggle/working/top100_final_recommendations.parquet


In [11]:
# =============================================================================
# CELL 11: Đánh giá trên User chẵn
# =============================================================================
print("Đang nạp tập Test và tạo danh sách đối chiếu (Ground Truth)... ")
df_test = pl.read_parquet(TEST_PATH)
truth_df = df_test.group_by('mapped_user_id').agg(pl.col('mapped_item_id').alias('true_items'))
n_valid = truth_df.height
print(f"Tổng số người dùng có tương tác trong tập Test: {n_valid:,}")
# Đánh giá trên User CHẴN (khớp với tập train)
truth_df = truth_df.filter((pl.col('mapped_user_id') % 2) == 0)
n_valid_even = truth_df.select('mapped_user_id').n_unique()
print(f"Số lượng người dùng chẵn (Even Users) dùng để đánh giá: {n_valid_even:,}")
print("Đang tính toán chỉ số bằng luồng dữ liệu (Streaming Evaluation)...\n")
pf = pq.ParquetFile(FINAL_TOP100_PATH)
reader = pf.iter_batches(batch_size=5000000, columns=['mapped_user_id', 'mapped_item_id', 'score'])
hr_sum = 0
ndcg_sum = 0.0
rc100_sum = 0
hr100_sum = 0
rc10_sum = 0.0
p10_sum = 0.0
buffer_df = pl.DataFrame()
idcg_lookup = [0.0] + [sum(1.0 / np.log2(x + 1) for x in range(1, k + 1)) for k in range(1, 11)]
def process_completed(df_chunk, truth_df_local):
    global hr_sum, ndcg_sum, rc100_sum, hr100_sum, rc10_sum, p10_sum
    
    df_chunk = df_chunk.with_columns(
        pl.col('score').rank(method='ordinal', descending=True).over('mapped_user_id').alias('rank')
    )
    top100 = df_chunk.filter(pl.col('rank') <= 100)
    eval_df_100 = top100.join(truth_df_local, on='mapped_user_id', how='inner')
    if eval_df_100.height == 0:
        return
        
    eval_df_100 = eval_df_100.with_columns(
        pl.col('true_items').list.contains(pl.col('mapped_item_id')).alias('is_hit')
    )
    
    hits_100 = eval_df_100.filter(pl.col('is_hit'))
    if hits_100.height > 0:
        hr100_sum += hits_100.select('mapped_user_id').n_unique()
        
        user_hits_100 = hits_100.group_by('mapped_user_id').agg(pl.len().alias('hits'))
        user_true_len = eval_df_100.group_by('mapped_user_id').agg(pl.col('true_items').first().list.len().alias('true_len'))
        user_recall = user_true_len.join(user_hits_100, on='mapped_user_id', how='left').with_columns(
            pl.col('hits').fill_null(0).alias('hits')
        ).with_columns(
            (pl.col('hits') / pl.col('true_len')).alias('recall')
        )
        rc100_sum += user_recall.select(pl.col('recall').sum()).item()
        
        hits_10 = hits_100.filter(pl.col('rank') <= 10)
        if hits_10.height > 0:
            hr_sum += hits_10.select('mapped_user_id').n_unique()
            
            user_hits_10 = hits_10.group_by('mapped_user_id').agg(pl.len().alias('hits_10'))
            user_metrics_10 = user_true_len.join(user_hits_10, on='mapped_user_id', how='left').with_columns(
                pl.col('hits_10').fill_null(0).alias('hits_10')
            ).with_columns([
                (pl.col('hits_10') / pl.col('true_len')).alias('recall_10'),
                (pl.col('hits_10') / 10.0).alias('precision_10')
            ])
            rc10_sum += user_metrics_10.select(pl.col('recall_10').sum()).item()
            p10_sum += user_metrics_10.select(pl.col('precision_10').sum()).item()
            
            hits_10 = hits_10.with_columns(
                (1.0 / np.log2(pl.col('rank') + 1)).alias('dcg_term')
            )
            user_dcg = hits_10.group_by('mapped_user_id').agg(
                pl.col('dcg_term').sum().alias('user_dcg')
            )
            user_idcg = user_true_len.with_columns(
                pl.col('true_len').clip(upper_bound=10).map_elements(lambda k: idcg_lookup[k], return_dtype=pl.Float64).alias('user_idcg')
            )
            user_ndcg = user_idcg.join(user_dcg, on='mapped_user_id', how='left').with_columns(
                pl.col('user_dcg').fill_null(0.0)
            ).with_columns(
                (pl.col('user_dcg') / pl.col('user_idcg')).alias('user_ndcg')
            )
            ndcg_sum += user_ndcg.select(pl.col('user_ndcg').sum()).item()
for batch in tqdm(reader, desc="Evaluating Chunks"):
    chunk = pl.from_arrow(batch)
    if buffer_df.height > 0:
        chunk = pl.concat([buffer_df, chunk])
    last_user = chunk.get_column('mapped_user_id')[-1]
    completed = chunk.filter(pl.col('mapped_user_id') != last_user)
    buffer_df = chunk.filter(pl.col('mapped_user_id') == last_user)
    if completed.height > 0:
        process_completed(completed, truth_df)
    del chunk, completed
    gc.collect()
if buffer_df.height > 0:
    process_completed(buffer_df, truth_df)
HR10 = hr_sum / n_valid_even
NDCG10 = ndcg_sum / n_valid_even
RC100 = rc100_sum / n_valid_even
HR100 = hr100_sum / n_valid_even
RC10 = rc10_sum / n_valid_even
P10 = p10_sum / n_valid_even
print("="*55)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH FINALMLP (BPR + RE-RANKING)")
print(f"Tập người dùng đánh giá (Even Users): {n_valid_even:,}")
print(f"Recall @ 100 (RC@100):              {RC100:.4f}")
print(f"Hit Rate @ 100 (HR@100):            {HR100:.4f}")
print(f"Recall @ 10 (RC@10):                {RC10:.4f}")
print(f"Precision @ 10 (P@10):              {P10:.4f}")
print(f"Hit Rate @ 10 (HR@10):              {HR10:.4f}")
print(f"NDCG @ 10:                          {NDCG10:.4f}")
print("="*55)

Đang nạp tập Test và tạo danh sách đối chiếu (Ground Truth)... 
Tổng số người dùng có tương tác trong tập Test: 1,031,356
Số lượng người dùng chẵn (Even Users) dùng để đánh giá: 515,331
Đang tính toán chỉ số bằng luồng dữ liệu (Streaming Evaluation)...



Evaluating Chunks: 0it [00:00, ?it/s]

KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH FINALMLP (BPR + RE-RANKING)
Tập người dùng đánh giá (Even Users): 515,331
Recall @ 100 (RC@100):              0.0443
Hit Rate @ 100 (HR@100):            0.1012
Recall @ 10 (RC@10):                0.0171
Precision @ 10 (P@10):              0.0042
Hit Rate @ 10 (HR@10):              0.0410
NDCG @ 10:                          0.0166
